In [10]:
# Cell 0 — Project Imports

import torch
from torch.nn import functional as F

In [11]:
# Cell 1 — Voxel-Wise Cross-Entropy

def compute_voxelwise_cross_entropy(
    logits: torch.Tensor,  # [B, K, D, H, W], torch.float32
    target: torch.Tensor,  # [B, D, H, W], torch.long
) -> torch.Tensor:         # [B, D, H, W], torch.float32
    """각 voxel의 정답 class에 대한 Cross-Entropy 계산."""

    # 각 voxel의 정답 class에 대한 loss 계산
    # reduction="none"으로 spatial dimension 보존
    voxelwise_cross_entropy = F.cross_entropy(
        input=logits,
        target=target,
        reduction="none",
    )  # [B, D, H, W]

    return voxelwise_cross_entropy


# 3개 class, 4개 voxel의 가상 logits 생성
synthetic_logits = torch.tensor(
    [
        [
            [[[4.0, 0.0], [0.0, 0.0]]],  # Class 0 logits
            [[[0.0, 4.0], [0.0, 2.0]]],  # Class 1 logits
            [[[0.0, 0.0], [4.0, 1.0]]],  # Class 2 logits
        ]
    ],
    dtype=torch.float32,
)  # [B=1, K=3, D=1, H=2, W=2]


# 각 voxel의 정답 class ID 생성
synthetic_target = torch.tensor(
    [[[[0, 1], [2, 1]]]],
    dtype=torch.long,
)  # [B=1, D=1, H=2, W=2]


# Class 축 K에 Softmax를 적용해 probability 계산
class_probabilities = synthetic_logits.softmax(
    dim=1,
)  # [B=1, K=3, D=1, H=2, W=2]


# Target class ID에 해당하는 probability만 선택
target_probabilities = class_probabilities.gather(
    dim=1,
    index=synthetic_target.unsqueeze(dim=1),
).squeeze(
    dim=1,
)  # [B=1, D=1, H=2, W=2]


# PyTorch의 voxel별 Cross-Entropy 계산
voxelwise_cross_entropy = compute_voxelwise_cross_entropy(
    logits=synthetic_logits,
    target=synthetic_target,
)  # [B=1, D=1, H=2, W=2]


# Cross-Entropy 정의를 이용한 직접 계산
manual_cross_entropy = -torch.log(
    target_probabilities,
)  # [B=1, D=1, H=2, W=2]


# 모든 voxel의 loss를 평균해 scalar 생성
mean_cross_entropy = voxelwise_cross_entropy.mean()  # []


print("Logits shape:        ", synthetic_logits.shape)
print("Target shape:        ", synthetic_target.shape)
print("Target probabilities:", target_probabilities)
print("Voxel-wise CE:       ", voxelwise_cross_entropy)
print("Manual CE:           ", manual_cross_entropy)
print("Mean CE:             ", mean_cross_entropy.item())

print(
    "Manual match:        ",
    torch.allclose(
        voxelwise_cross_entropy,
        manual_cross_entropy,
    ),
)

Logits shape:         torch.Size([1, 3, 1, 2, 2])
Target shape:         torch.Size([1, 1, 2, 2])
Target probabilities: tensor([[[[0.9647, 0.9647],
          [0.9647, 0.6652]]]])
Voxel-wise CE:        tensor([[[[0.0360, 0.0360],
          [0.0360, 0.4076]]]])
Manual CE:            tensor([[[[0.0360, 0.0360],
          [0.0360, 0.4076]]]])
Mean CE:              0.1288837492465973
Manual match:         True


In [12]:
# Cell 2 — Target One-Hot과 Soft Dice

def convert_target_to_one_hot(
    target: torch.Tensor,       # [B, D, H, W], torch.long
    number_of_classes: int,
    output_dtype: torch.dtype,
) -> torch.Tensor:              # [B, K, D, H, W], output_dtype
    """Class ID target을 class별 one-hot mask로 변환."""
    
    # 각 voxel의 class ID를 one-hot vector로 변환
    target_one_hot_last = F.one_hot(
        target,
        num_classes=number_of_classes,
    )  # [B, D, H, W, K], torch.long
    
    target_one_hot = target_one_hot_last.movedim(
        -1,
        1,  
    ).to(
        dtype=output_dtype,
    )  # [B, K, D, H, W], output_dtype
    
    return target_one_hot


def compute_soft_dice_per_case_and_class(
    probabilities: torch.Tensor,  # [B, K, D, H, W], floating point
    target_one_hot: torch.Tensor,  # [B, K, D, H, W], floating point
    smoothing: float = 1e-5,
) -> torch.Tensor:                 # [B, K], floating point
    """Case별·class별 Soft Dice 계산."""
    
    # spatial dimensions만 합산
    spatial_dimensions: tuple[int, int, int] = (
        2,
        3,
        4,
    )
    
    # Probability와 one-hot target이 겹치는 soft intersection 계산
    intersection = (
        probabilities
        * target_one_hot
    ).sum(
        dim=spatial_dimensions,
    ) # [B, K]
    
    # 각 case와 class의 predicted soft volume 계산
    predicted_mass = probabilities.sum(
        dim=spatial_dimensions,
    )  # [B, K]

    # 각 case와 class의 target voxel volume 계산
    target_mass = target_one_hot.sum(
        dim=spatial_dimensions,
    )  # [B, K]

    # Case별·class별 Soft Dice 계산
    soft_dice = (
        2.0 * intersection
        + smoothing
    ) / (
        predicted_mass
        + target_mass
        + smoothing
    )  # [B, K]
    
    return soft_dice


# Target class ID를 class별 one-hot mask로 변환
synthetic_target_one_hot = convert_target_to_one_hot(
    target=synthetic_target,
    number_of_classes=synthetic_logits.shape[1],
    output_dtype=class_probabilities.dtype,
)  # [B=1, K=3, D=1, H=2, W=2]


# Argmax 없이 probability를 사용해 Soft Dice 계산
soft_dice_per_case_and_class = (
    compute_soft_dice_per_case_and_class(
        probabilities=class_probabilities,
        target_one_hot=synthetic_target_one_hot,
    )
)  # [B=1, K=3]


print(
    "Target shape:        ",
    synthetic_target.shape,
)

print(
    "One-hot shape:       ",
    synthetic_target_one_hot.shape,
)

print(
    "Probability shape:   ",
    class_probabilities.shape,
)

print(
    "One-hot class masses:",
    synthetic_target_one_hot.sum(
        dim=(2, 3, 4),
    ),
)

print(
    "Soft Dice [B, K]:    ",
    soft_dice_per_case_and_class,
)

Target shape:         torch.Size([1, 1, 2, 2])
One-hot shape:        torch.Size([1, 3, 1, 2, 2])
Probability shape:    torch.Size([1, 3, 1, 2, 2])
One-hot class masses: tensor([[1., 2., 1.]])
Soft Dice [B, K]:     tensor([[0.9231, 0.8894, 0.8595]])


In [13]:
# Cell 3 — Foreground Soft Dice Reduction

def reduce_foreground_soft_dice(
    soft_dice_per_case_and_class: torch.Tensor,  # [B, K]
    background_class_id: int = 0,
) -> tuple[
    torch.Tensor,  # Foreground Dice [B, K-1]
    torch.Tensor,  # Case별 foreground macro Dice [B]
    torch.Tensor,  # Foreground Soft Dice loss []
]:
    """Background 제외 후 case·class 평균과 Dice loss 계산."""
    
    # 전체 class ID 생성
    class_ids = torch.arange(
        soft_dice_per_case_and_class.shape[1],
        device=soft_dice_per_case_and_class.device,
    )  # [K]
    
    # Background가 아닌 class 위치 선택
    foreground_class_mask = (
        class_ids != background_class_id
    )  # [K], torch.bool

    # Background Dice를 제외하고 foreground class만 선택
    foreground_soft_dice = (
        soft_dice_per_case_and_class[
            :,
            foreground_class_mask,
        ]
    )  # [B, K-1]
    
    # 각 case에서 foreground class Dice의 macro average 계산
    case_foreground_macro_dice = (
        foreground_soft_dice.mean(
            dim=1,
        )
    )  # [B]
    
    # 모든 case를 동일한 비중으로 평균
    mean_foreground_soft_dice = (
        case_foreground_macro_dice.mean()
    )  # []

    # 최대화할 Dice를 최소화할 loss로 변환
    foreground_soft_dice_loss = (
        1.0
        - mean_foreground_soft_dice
    )  # []

    return (
        foreground_soft_dice,
        case_foreground_macro_dice,
        foreground_soft_dice_loss,
    )
    
    
    # Background까지 포함한 평균 계산
mean_dice_including_background = (
    soft_dice_per_case_and_class.mean()
)  # []


# Background를 제외한 foreground Dice와 loss 계산
(
    foreground_soft_dice,
    case_foreground_macro_dice,
    foreground_soft_dice_loss,
) = reduce_foreground_soft_dice(
    soft_dice_per_case_and_class=(
        soft_dice_per_case_and_class
    ),
    background_class_id=0,
)


print(
    "All-class Dice [B, K]:       ",
    soft_dice_per_case_and_class,
)

print(
    "Foreground Dice [B, K-1]:    ",
    foreground_soft_dice,
)

print(
    "Case foreground macro [B]:   ",
    case_foreground_macro_dice,
)

print(
    "Mean including background:   ",
    mean_dice_including_background.item(),
)

print(
    "Foreground Soft Dice loss []:",
    foreground_soft_dice_loss.item(),
)

All-class Dice [B, K]:        tensor([[0.9231, 0.8894, 0.8595]])
Foreground Dice [B, K-1]:     tensor([[0.8894, 0.8595]])
Case foreground macro [B]:    tensor([0.8744])
Mean including background:    0.8906622529029846
Foreground Soft Dice loss []: 0.12556123733520508


In [14]:
# Cell 4 — Cross-Entropy와 Soft Dice 결합

def compute_cross_entropy_soft_dice_loss(
    logits: torch.Tensor,  # [B, K, D, H, W], floating point
    target: torch.Tensor,  # [B, D, H, W],    torch.long
    cross_entropy_weight: float = 1.0,
    soft_dice_weight: float = 1.0,
    smoothing: float = 1e-5,
    background_class_id: int = 0,
) -> tuple[
    torch.Tensor,  # Cross-Entropy loss []
    torch.Tensor,  # Foreground Soft Dice loss []
    torch.Tensor,  # Combined loss []
]:
    """Cross-Entropy와 foreground Soft Dice의 가중합 계산."""

    # Raw logits와 class ID target으로 Cross-Entropy 계산 (Background를 포함)
    cross_entropy_loss = F.cross_entropy(
        input=logits,
        target=target,
        reduction="mean",
    )  # []
    
    # Class 축 K에서 continuous probability 계산
    probabilities = logits.softmax(
        dim=1,
    )  # [B, K, D, H, W]

    # Class ID target을 probability와 동일한 one-hot Shape로 변환
    target_one_hot = convert_target_to_one_hot(
        target=target,
        number_of_classes=logits.shape[1],
        output_dtype=probabilities.dtype,
    )  # [B, K, D, H, W]
    
    # Case별·class별 Soft Dice 계산
    soft_dice_per_case_and_class = (
        compute_soft_dice_per_case_and_class(
            probabilities=probabilities,
            target_one_hot=target_one_hot,
            smoothing=smoothing,
        )
    )  # [B, K]
    
    # Background 제외 후 foreground Dice loss 계산
    (
        _,
        _,
        foreground_soft_dice_loss,
    ) = reduce_foreground_soft_dice(
        soft_dice_per_case_and_class=(
            soft_dice_per_case_and_class
        ),
        background_class_id=background_class_id,
    )  # []
    
    # 두 scalar loss의 weighted sum 계산
    combined_loss = (
        cross_entropy_weight
        * cross_entropy_loss
        + soft_dice_weight
        * foreground_soft_dice_loss
    )  # []

    return (
        cross_entropy_loss,
        foreground_soft_dice_loss,
        combined_loss,
    )
    

# 동일한 synthetic logits와 target으로 combined loss 계산
(
    cross_entropy_loss,
    foreground_soft_dice_loss,
    combined_loss,
) = compute_cross_entropy_soft_dice_loss(
    logits=synthetic_logits,
    target=synthetic_target,
    cross_entropy_weight=1.0,
    soft_dice_weight=1.0,
    smoothing=1e-5,
    background_class_id=0,
)


print(
    "Cross-Entropy loss []:      ",
    cross_entropy_loss.item(),
)

print(
    "Foreground Dice loss []:    ",
    foreground_soft_dice_loss.item(),
)

print(
    "Combined loss []:           ",
    combined_loss.item(),
)

print(
    "Component sum matches:      ",
    torch.allclose(
        combined_loss,
        (
            cross_entropy_loss
            + foreground_soft_dice_loss
        ),
    ),
)

Cross-Entropy loss []:       0.1288837492465973
Foreground Dice loss []:     0.12556123733520508
Combined loss []:            0.25444498658180237
Component sum matches:       True


In [15]:
# Cell 5 — Combined Loss Gradient Sanity Check

# 기존 logits와 독립된 gradient 추적용 leaf Tensor 생성
trainable_logits = (
    synthetic_logits
    .detach()
    .clone()
    .requires_grad_(True)
)  # [B=1, K=3, D=1, H=2, W=2]


# Gradient 추적이 활성화된 combined loss 계산
(
    trainable_cross_entropy_loss,
    trainable_soft_dice_loss,
    trainable_combined_loss,
) = compute_cross_entropy_soft_dice_loss(
    logits=trainable_logits,
    target=synthetic_target,
    cross_entropy_weight=1.0,
    soft_dice_weight=1.0,
    smoothing=1e-5,
    background_class_id=0,
)

# Combined loss에서 logits 방향으로 gradient 계산
trainable_combined_loss.backward()


# Leaf logits에 저장된 gradient 확인
logits_gradient = trainable_logits.grad

if logits_gradient is None:
    raise RuntimeError(
        "Combined loss에서 logits로 gradient가 전달되지 않았습니다."
    )


# Loss와 gradient의 수치 안정성 확인
loss_is_finite = bool(
    torch.isfinite(
        trainable_combined_loss
    ).item()
)

gradient_is_finite = bool(
    torch.isfinite(
        logits_gradient
    ).all().item()
)

# 실제 parameter update 신호가 존재하는지 확인
has_nonzero_gradient = bool(
    (
        logits_gradient.abs() > 0
    ).any().item()
)

# 전체 gradient 크기 계산
gradient_norm = logits_gradient.norm()  # []


# 작은 gradient descent step 적용
learning_rate: float = 0.5

with torch.no_grad():
    updated_logits = (
        trainable_logits
        - learning_rate
        * logits_gradient
    )  # [B=1, K=3, D=1, H=2, W=2]


# Update 이후 combined loss 재계산
(
    _,
    _,
    updated_combined_loss,
) = compute_cross_entropy_soft_dice_loss(
    logits=updated_logits,
    target=synthetic_target,
    cross_entropy_weight=1.0,
    soft_dice_weight=1.0,
    smoothing=1e-5,
    background_class_id=0,
)


# Gradient descent 방향에서 loss 감소 여부 확인
loss_decreased_after_step = bool(
    (
        updated_combined_loss
        < trainable_combined_loss
    ).item()
)


print(
    "Combined loss shape:    ",
    trainable_combined_loss.shape,
)

print(
    "Gradient shape:         ",
    logits_gradient.shape,
)

print(
    "Loss is finite:         ",
    loss_is_finite,
)

print(
    "Gradient is finite:     ",
    gradient_is_finite,
)

print(
    "Has non-zero gradient:  ",
    has_nonzero_gradient,
)

print(
    "Gradient norm:          ",
    gradient_norm.item(),
)

print(
    "Original loss:          ",
    trainable_combined_loss.item(),
)

print(
    "Updated loss:           ",
    updated_combined_loss.item(),
)

print(
    "Loss decreased:         ",
    loss_decreased_after_step,
)

Combined loss shape:     torch.Size([])
Gradient shape:          torch.Size([1, 3, 1, 2, 2])
Loss is finite:          True
Gradient is finite:      True
Has non-zero gradient:   True
Gradient norm:           0.19712533056735992
Original loss:           0.25444498658180237
Updated loss:            0.23565372824668884
Loss decreased:          True
